<!--<badge>--><a href="https://colab.research.google.com/github/JoeChen322/Fintech/blob/main/BC_2_Group17.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a><!--</badge>-->


## Table of Contents

1. [Imports](#imports)
2. [EDA](#eda)


# Imports

This function block allows you to import the data tables present on the shared GitHub repository


In [ ]:
from typing import Dict, List, Optional, Tuple

import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import tqdm
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from tabulate import tabulate
from xgboost import XGBClassifier

%config InlineBackend.figure_format = 'retina'


In [2]:
url = "https://drive.google.com/uc?id=1TiR645keulkG4pONLpVn-bO4elNBtNpX"
file_path = "Dataset2_Needs.xls"
gdown.download(url, file_path, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1TiR645keulkG4pONLpVn-bO4elNBtNpX
To: /Users/pengrao/Workspace/Fintech/Dataset2_Needs.xls
100%|██████████| 428k/428k [00:00<00:00, 10.3MB/s]


'Dataset2_Needs.xls'

The next cells aim to load the Excel workbook and separate the sheets into:

- the main client dataset : needs_df
- the products catalogue : products_df
- the metadata table : metadata_df

Initially, we will analyze the need_df table containing the data of the different clients. We are going to inspect structure and variable meaning of this specific table.


In [3]:
needs_df = pd.read_excel("Dataset2_Needs.xls", sheet_name="Needs")
products_df = pd.read_excel("Dataset2_Needs.xls", sheet_name="Products")

# The metadata retrieved is that relating to the needs_df database
metadata_df = pd.read_excel("Dataset2_Needs.xls", sheet_name="Metadata", nrows=11)

# We delete the first line, which doesn't serve much purpose
metadata_df = metadata_df[metadata_df["Metadata"] != "Clients"]

# We remove any potentially unnecessary spaces at the beginning and end of each column name.
needs_df.columns = needs_df.columns.str.strip()
products_df.columns = products_df.columns.str.strip()
metadata_df.columns = metadata_df.columns.str.strip()

# List of categorical columns in the dataset:
CATEGORICAL_COLS = ["Gender"]

# List of numeric columns in the dataset:
NUMERIC_COLS = [
    "Age",
    "FamilyMembers",
    "FinancialEducation",
    "RiskPropensity",
    "Income",
    "Wealth",
]

# List of target columns in the dataset:
TARGET_COLS = ["IncomeInvestment", "AccumulationInvestment"]

# Value mapping: introduces dictionaries to convert numeric codes into readable labels
VALUE_MAPS: Dict[str, Dict[int, str]] = {
    "Gender": {0: "Male", 1: "Female"},
    "AccumulationInvestment": {0: "Low propensity", 1: "High propensity"},
    "IncomeInvestment": {0: "Low propensity", 1: "High propensity"},
}

needs_df.head()

,ID,Age,Gender,FamilyMembers,FinancialEducation,RiskPropensity,Income,Wealth,IncomeInvestment,AccumulationInvestment
0,1,60,0,2,0.228685,0.233355,68.181525,53.260067,0,1
1,2,78,0,2,0.358916,0.170911,21.807595,135.550048,1,0
2,3,33,1,2,0.317515,0.249703,23.252747,66.303678,0,1
3,4,69,1,4,0.767685,0.654597,166.189034,404.997689,1,1
4,5,58,0,3,0.429719,0.349039,21.186723,58.911930,0,0


# EDA


# Prediction Model ML


## Feature Engineering


In [4]:
# Step 1: Feature engineering and transformation function
def prepare_features(df):
    X = df.copy()

    # Log transformation for Wealth and Income
    X["Wealth_log"] = np.log1p(X["Wealth"])
    X["Income_log"] = np.log1p(X["Income"])

    # Create Income/Wealth ratio
    X["Income_Wealth_Ratio"] = (
        X["Income"].div(X["Wealth"].replace(0, np.nan)).fillna(X["Income"].max())
    )
    X["Income_Wealth_Ratio_log"] = np.log1p(X["Income_Wealth_Ratio"])

    # Select features for modeling
    features_base = [
        "Age",
        "Gender",
        "FamilyMembers",
        "FinancialEducation",
        "RiskPropensity",
        "Wealth_log",
        "Income_log",
    ]

    features_engineered = [
        "Age",
        "Gender",
        "FamilyMembers",
        "FinancialEducation",
        "RiskPropensity",
        "Income_Wealth_Ratio_log",
    ]

    # Normalize all features
    scaler = MinMaxScaler()
    X_base = pd.DataFrame(scaler.fit_transform(X[features_base]), columns=features_base)
    X_engineered = pd.DataFrame(
        scaler.fit_transform(X[features_engineered]), columns=features_engineered
    )

    return X_base, X_engineered


# Step 2: Data split function
def split_data(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    return X_train, X_test, y_train, y_test


# Step 3: Model training and evaluation function
def train_evaluate_model(X_train, y_train, X_test, y_test, model, k_folds=5):
    kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)
    cv_metrics = {"accuracy": [], "precision": [], "recall": [], "f1": []}

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
        X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model.fit(X_train_fold, y_train_fold)
        y_val_pred = model.predict(X_val_fold)

        cv_metrics["accuracy"].append(accuracy_score(y_val_fold, y_val_pred))
        cv_metrics["precision"].append(precision_score(y_val_fold, y_val_pred))
        cv_metrics["recall"].append(recall_score(y_val_fold, y_val_pred))
        cv_metrics["f1"].append(f1_score(y_val_fold, y_val_pred))

    model.fit(X_train, y_train)
    y_test_pred = model.predict(X_test)

    return {
        "cv_metrics": {
            metric: {"mean": np.mean(scores), "std": np.std(scores)}
            for metric, scores in cv_metrics.items()
        },
        "test_metrics": {
            "accuracy": accuracy_score(y_test, y_test_pred),
            "precision": precision_score(y_test, y_test_pred),
            "recall": recall_score(y_test, y_test_pred),
            "f1": f1_score(y_test, y_test_pred),
        },
    }


# Step 4: Display results function
def display_results_table(results_dict, model_name, feature_type):
    cv_data = {
        "Metric": ["Accuracy", "Precision", "Recall", "F1"],
        "CV Mean": [
            results_dict["cv_metrics"]["accuracy"]["mean"],
            results_dict["cv_metrics"]["precision"]["mean"],
            results_dict["cv_metrics"]["recall"]["mean"],
            results_dict["cv_metrics"]["f1"]["mean"],
        ],
        "CV Std": [
            results_dict["cv_metrics"]["accuracy"]["std"],
            results_dict["cv_metrics"]["precision"]["std"],
            results_dict["cv_metrics"]["recall"]["std"],
            results_dict["cv_metrics"]["f1"]["std"],
        ],
        "Test Set": [
            results_dict["test_metrics"]["accuracy"],
            results_dict["test_metrics"]["precision"],
            results_dict["test_metrics"]["recall"],
            results_dict["test_metrics"]["f1"],
        ],
    }

    df = pd.DataFrame(cv_data)
    df = df.round(3)

    print(f"\n{model_name} - {feature_type}")
    print("=" * 60)
    print(tabulate(df, headers="keys", tablefmt="pretty"))

## Random Forest


In [5]:
X_base, X_engineered = prepare_features(needs_df)
y_income = needs_df["IncomeInvestment"]
y_accum = needs_df["AccumulationInvestment"]

X_train, X_test, y_train, y_test = split_data(X_base, y_accum)

In [6]:
# Implement Random Forest for client needs classification
# Data is already prepared (X_train, X_test, y_train, y_test from previous cells)

# Train a default Random Forest (no hyperparameter tuning)
rf_model = RandomForestClassifier()
rf_model.fit(X_train, y_train)

# Predictions
y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]

# Model evaluation
print("\n" + "=" * 50)
print("RANDOM FOREST MODEL EVALUATION")
print("=" * 50)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

# ROC-AUC Score
roc_auc_rf = roc_auc_score(y_test, y_pred_proba_rf)
print(f"\nROC-AUC Score: {roc_auc_rf:.4f}")

# Cross-validation scores
cv_scores_rf = cross_val_score(rf_model, X_train, y_train, cv=5, scoring="roc_auc")
print(f"Cross-validation ROC-AUC scores: {cv_scores_rf}")
print(f"Mean CV ROC-AUC: {cv_scores_rf.mean():.4f} (+/- {cv_scores_rf.std():.4f})")


RANDOM FOREST MODEL EVALUATION

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.82      0.79       487
           1       0.81      0.75      0.78       513

    accuracy                           0.78      1000
   macro avg       0.78      0.78      0.78      1000
weighted avg       0.78      0.78      0.78      1000


Confusion Matrix:
[[399  88]
 [130 383]]

ROC-AUC Score: 0.8434
Cross-validation ROC-AUC scores: [0.84866055 0.83138498 0.85328905 0.82958412 0.84789243]
Mean CV ROC-AUC: 0.8422 (+/- 0.0097)


# Prediction Model DL


# Recommendation System
